Generate: A model produces an answer using a prompt that includes both the question with the retrieved data

![alt text](rag_retrieval_generation.avif)

1. What is Generation?

Generation means the LLM creates the final response using the retrieved context and the user query.

1. Receive User Query
2. Receive Retrieved Context -The retriever returns the most relevant chunks.
3. Chaining   -prompt|llm|retriver


Chaining means connecting multiple components so the output of one step becomes the input of the next step.

This pipe symbol means “pass output to the next step.

retriever | prompt | llm

| Component            | Purpose                  |
| -------------------- | ------------------------ |
| `retriever`          | finds relevant documents |
| `ChatPromptTemplate` | builds the prompt        |
| `llm`                | generates answer         |
| `StrOutputParser`    | converts output to text  |
| `chain`              | connects everything      |


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS


loader = TextLoader(r"doc/data.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)



In [10]:
chunks = splitter.split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = FAISS.from_documents(chunks , embeddings)
vectorstore.add_documents(chunks)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})

In [8]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from langchain_openai import ChatOpenAI
# 1. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# Prompt Template
prompt = ChatPromptTemplate.from_template(
"""
Answer the question using the context below.if not know thwn tell dony know

Context:
{context}

Question:
{question}
"""
)

rag_chain = ({"context":
    retriever,"question": RunnablePassthrough()}
|prompt
| llm 
)


In [23]:
response=rag_chain.invoke("types of plant")
print(response)


The types of plants mentioned in the context are:

- Trees: Large plants with a single woody stem or trunk.
- Shrubs: Woody plants smaller than trees, typically with multiple stems.
- Herbs: Non-woody plants with soft stems.
- Grasses: Monocotyledonous plants with narrow leaves.
- Ferns: Plants that reproduce via spores, not seeds.
- Succulents/Cacti: Plants with thick, fleshy parts for water storage.
- Aquatic Plants: Plants adapted to living in water.
